In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dask.dataframe as dd
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import TomekLinks
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV

In [ ]:
!pip install dask

In [ ]:
X = dd.read_csv("x_70p_pca.csv").compute()
y = dd.read_csv("is_fraud.csv").compute()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
tomek = TomekLinks()
X_tl, y_tl = tomek.fit_resample(X = X_train, y = y_train)

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris

# Train a Random Forest model
model = RandomForestClassifier(random_state=42, criterion = "gini", max_depth = 3, max_features = 9, min_samples_leaf = 7, min_samples_split = 6)
model.fit(X_train, y_train)

# Initialize JS visualization code
shap.initjs()

# Create a SHAP Tree Explainer
explainer = shap.TreeExplainer(model)

# Calculate SHAP values - this might take a bit for larger models
shap_values = explainer.shap_values(X)

# Plot the partial dependence plot for the first feature
shap.dependence_plot(0, shap_values[1], X, feature_names=X.columns, interaction_index=None)

# If you want to save the plot
plt.savefig('partial_dependence_plot.png')


In [ ]:
clf = DecisionTreeClassifier(random_state = 42)
clf.fit(X_tl, y_tl)

y_pred = clf.predict(X_test)

accuracy_best = accuracy_score(y_test, y_pred)
# Print the accuracy of the model with the best hyperparameters
print("Accuracy with best hyperparameters:", accuracy_best)

print("Accuracy:", accuracy_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Precision
precision = precision_score(y_test, y_pred)
print("Precision for fraud:", precision)

# Recall
recall = recall_score(y_test, y_pred)
print("Recall for fraud:", recall)

# F1-Score
f1 = f1_score(y_test, y_pred)
print("F1-Score for fraud:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/merged_df.csv', low_memory = False, on_bad_lines = "skip")
df = df.dropna(subset=['isFraud'])
df= df.drop(columns=['Num'])

y = df['isFraud']
df = df.drop('isFraud', axis=1)


for column in df.columns:
  most_common_value = df[column].mode()[0]
  df[column] = df[column].fillna(most_common_value)

In [ ]:
print(y[y == 1], y[y == 0])

In [ ]:
df.drop_duplicates(inplace = True)

In [ ]:
"""def remove_outliers(df, columns_to_check):

  This function removes outliers from specific columns in a DataFrame based on given criteria.

  Args:
      df (pd.DataFrame): The DataFrame to clean.
      columns_to_check (list): A list of column names to check for outliers.

  Returns:
      pd.DataFrame: The cleaned DataFrame with outliers removed from specified columns


  for column in columns_to_check:
    # Calculate the interquartile range (IQR)
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    # Define the lower and upper bounds for outlier removal
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify and remove outliers
    outlier_indices = df[((df[column] < lower_bound) | (df[column] > upper_bound))].index
    df = df.drop(outlier_indices)
  return df

columns_to_check =[col for col in df.columns if df[col].dtype!="object"]
df_cleaned = remove_outliers(df, columns_to_check)
print(df_cleaned)"""

In [ ]:
# Selecting numerical and categorical data
num_df = df.select_dtypes(include=['int64', 'float64'])
cat_df = df.select_dtypes(include=['object', 'category'])

# Initialize a label encoder for each categorical column in cat_df
for column in cat_df.columns:
    le = LabelEncoder()
    cat_df[column] = le.fit_transform(cat_df[column])

# Concatenate the numerical data and the newly encoded categorical data
final_df = pd.concat([num_df.reset_index(drop=True), cat_df.reset_index(drop=True)], axis=1)

In [ ]:

# Selecting numerical and categorical data
num_df = df.select_dtypes(include=['int64', 'float64'])
cat_df = df.select_dtypes(include=['object', 'category'])

# Initialize and fit the OneHotEncoder
encoder = OneHotEncoder(sparse=False)  # Correct parameter name is 'sparse'
cat_df_encoded = encoder.fit_transform(cat_df)  # Fit and transform the categorical data

# Convert the encoded data into a DataFrame
cat_df_encoded = pd.DataFrame(cat_df_encoded, columns=encoder.get_feature_names_out(cat_df.columns))

# Concatenate the numerical data and the newly encoded categorical data
final_df = pd.concat([num_df.reset_index(drop=True), cat_df_encoded.reset_index(drop=True)], axis=1)

In [ ]:

missing_data = final_df[final_df.isnull().any(axis=1)]
print(missing_data)


In [ ]:
#X = final_df
X = pd.read_csv("/content/Xnew_data.csv")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X.to_csv("Xnew_data.csv", index = True)

In [ ]:
pca = PCA(n_components=0.95)  # 95% de variance
X_pca = pca.fit_transform(X_scaled)

X = pd.DataFrame(X_pca, columns=[f'PC_{i}' for i in range(X_pca.shape[1])])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
PCaclf = DecisionTreeClassifier(class_weight = {0:10, 1:1},random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy_best = accuracy_score(y_test, y_pred)
# Print the accuracy of the model with the best hyperparameters
print("Accuracy with best hyperparameters:", accuracy_best)

print("Accuracy:", accuracy_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Precision
precision = precision_score(y_test, y_pred)
print("Precision for fraud:", precision)

# Recall
recall = recall_score(y_test, y_pred)
print("Recall for fraud:", recall)

# F1-Score
f1 = f1_score(y_test, y_pred)
print("F1-Score for fraud:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)

In [ ]:
clf = DecisionTreeClassifier(class_weights = {0: 1, 1: 10},random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy_best = accuracy_score(y_test, y_pred)
# Print the accuracy of the model with the best hyperparameters
print("Accuracy with best hyperparameters:", accuracy_best)

print("Accuracy:", accuracy_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Precision
precision = precision_score(y_test, y_pred)
print("Precision for fraud:", precision)

# Recall
recall = recall_score(y_test, y_pred)
print("Recall for fraud:", recall)

# F1-Score
f1 = f1_score(y_test, y_pred)
print("F1-Score for fraud:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)

In [ ]:
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

accuracy_best = accuracy_score(y_test, y_pred)
# Print the accuracy of the model with the best hyperparameters
print("Accuracy with best hyperparameters:", accuracy_best)

print("Accuracy:", accuracy_score(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Precision
precision = precision_score(y_test, y_pred)
print("Precision for fraud:", precision)

# Recall
recall = recall_score(y_test, y_pred)
print("Recall for fraud:", recall)

# F1-Score
f1 = f1_score(y_test, y_pred)
print("F1-Score for fraud:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)

In [ ]:
# Define the grid of hyperparameters to search
param_grid = {
    'max_depth': [1, 3, 4, 5 ],
    'min_samples_leaf': [ 3, 4, 5, 8],
    'min_samples_split': [ 2, 3, 15, 20]
}

# Create the GridSearchCV object
grid_search = GridSearchCV(clf, param_grid, cv=5)

# Fit the GridSearchCV object to the data
grid_search.fit(X_train, y_train)

# Get the best hyperparameters
best_params = grid_search.best_params_

# Print the best hyperparameters
print("Best hyperparameters:")
print(best_params)

# Train the model with the best hyperparameters
clf_best = DecisionTreeClassifier(class_weight={0:1, 1:20},**best_params)
clf_best.fit(X_train, y_train)

# Evaluate the model on the test set
y_pred_best = clf_best.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)

# Print the accuracy of the model with the best hyperparameters
print("Accuracy with best hyperparameters:", accuracy_best)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
cm = confusion_matrix(y_test, y_pred_best)
print("Confusion Matrix:\n", cm)

# Precision
precision = precision_score(y_test, y_pred)
print("Precision for fraud:", precision)

# Recall
recall = recall_score(y_test, y_pred)
print("Recall for fraud:", recall)

# F1-Score
f1 = f1_score(y_test, y_pred)
print("F1-Score for fraud:", f1)

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC-AUC Score:", roc_auc)


In [ ]:
# prompt: give me a code that ensure the usage of the fonction of cost sensitive learning

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load the data
df = pd.read_csv('your_data.csv')

# Separate features and target
X = df.drop('isFraud', axis=1)
y = df['isFraud']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the cost-sensitive decision tree classifier
clf = DecisionTreeClassifier(class_weight={0: 1, 1: 10})

# Train the classifier
clf.fit(X_train, y_train)

# Make predictions on the test set
y_pred = clf.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy
print("Accuracy:", accuracy)
